## config_loader_py_test

In [7]:
import sys
from pathlib import Path

PIPELINE_ROOT = Path(r"C:\streaming_emulator")
if str(PIPELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(PIPELINE_ROOT))


In [9]:
from pathlib import Path
from kafka_metrics.app.config_loader import MetricsConsumerConfig


PROJECT_ROOT = Path.cwd().parent


config_path = PROJECT_ROOT / "kafka_metrics" / "config" / "metrics_consumer.json"

cfg = MetricsConsumerConfig(config_path)

print(cfg.kafka_bootstrap_servers)
print(cfg.kafka_topics)
print(cfg.kafka_group_id)
print(cfg.http_port)


localhost:9092
['telemetry.engine', 'telemetry.transmission', 'telemetry.battery', 'telemetry.tyre', 'telemetry.body']
kafka-metrics-consumer
9201


## schemas_py_test

In [11]:
import json

In [12]:
from kafka_metrics.app.schemas import ParsedKafkaMessage

payload = {
    "metadata": {
        "vehicle_id": "sim001",
        "module": "engine",
        "ingest_ts": "2026-01-10T12:00:00Z",
    },
    "data": {"rpm": 1200}
}

msg = ParsedKafkaMessage.from_kafka_value(
    value=json.dumps(payload).encode("utf-8")
)

print(msg.vehicle_id)
print(msg.module)
print(msg.ingest_ts)
print(msg.raw_json["data"])


sim001
engine
2026-01-10T12:00:00Z
{'rpm': 1200}


In [13]:
payload = {
    "metadata": {
        "vehicle_id": "sim002",
        "module": "battery",
    },
    "data": {"soc": 0.82}
}

msg = ParsedKafkaMessage.from_kafka_value(
    value=json.dumps(payload).encode("utf-8")
)

print(msg.ingest_ts)


None


In [14]:
payload = {"data": {"rpm": 1000}}

try:
    ParsedKafkaMessage.from_kafka_value(
        value=json.dumps(payload).encode("utf-8")
    )
except Exception as e:
    print(type(e).__name__, str(e))


KafkaMessageParseError Missing or invalid 'metadata' object


## state_py_test

In [15]:
from kafka_metrics.app.state import MetricsState

state = MetricsState(max_latency_samples=3)

state.record_message(
    vehicle_id="sim001",
    full_event={"a": 1},
    latency_ms=10,
)
state.record_message(
    vehicle_id="sim002",
    full_event={"b": 2},
    latency_ms=20,
)

assert state.total_rows() == 2
assert state.per_vehicle_counts()["sim001"] == 1
assert state.per_vehicle_counts()["sim002"] == 1


In [16]:
state = MetricsState(max_latency_samples=2)

state.record_message(vehicle_id="v", full_event={}, latency_ms=1)
state.record_message(vehicle_id="v", full_event={}, latency_ms=2)
state.record_message(vehicle_id="v", full_event={}, latency_ms=3)

assert state.latency_samples() == [2, 3]


In [17]:
state.reset()
assert state.total_rows() == 0
assert state.per_vehicle_counts() == {}
assert state.latest_all() == {}


## metrics_py_test

In [18]:
from kafka_metrics.app.metrics import kafka_rows_total
kafka_rows_total.inc()

In [19]:
from kafka_metrics.app.metrics import kafka_rows_per_vehicle

kafka_rows_per_vehicle.labels(vehicle_id="sim001").inc()
kafka_rows_per_vehicle.labels(vehicle_id="sim002").inc(3)

In [20]:
from kafka_metrics.app.metrics import kafka_processing_latency_ms

kafka_processing_latency_ms.observe(42)